In [ ]:
import sys

# Add the parent directory to the system path
sys.path.append("../04_survival_models/src")

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
from tqdm import tqdm
import itertools
import os
import matplotlib.pyplot as plt
from uc2_functions import (
    compare_random_states,
    plot_metric_boxplot,
    plot_features_violin,
)

In [ ]:
np.random.seed(42)
sns.set(style="whitegrid")

# Goal

The goal is to compare survival models over Monte Carlo simulations.

# Parameters

In [ ]:
DIR_GRANT_FINETUNE = "../02_survival_grant_finetune"
DIR_FEATURE_SELECTION = "../03_survival_feature_selection"
DIR_SURVIVAL_MODELS = "../04_survival_models"
DIR_ARTIFACTS = "artifacts"
DIR_FIGURES = "../figures/05_compare_models"
PATH_METRICS_GRANT_FINETUNE_LARCHER = (
    "df_metrics_UC2_larcher_survival_grant_finetune_2024_05.csv"
)
PATH_METRICS_GRANT_FINETUNE_RAW = (
    "df_metrics_UC2_raw_survival_grant_finetune_2024_05.csv"
)
PATH_METRICS_FEATURE_SELECTION_LARCHER = "df_metrics_UC2_larcher_2024_02.csv"
PATH_METRICS_FEATURE_SELECTION_RAW = "df_metrics_UC2_raw_2024_02.csv"
PATH_METRICS_SURVIVAL_MODELS = "df_metrics_UC2_raw_survival_models_2024_07.csv"
# Number of simulations
S = 100
os.makedirs(DIR_FIGURES, exist_ok=True)


# Data ingestion

## GRANT finetune

### Larcher dataset

In [ ]:
df_metrics_grant_finetune_larcher = pd.read_csv(
    os.path.join(DIR_GRANT_FINETUNE, DIR_ARTIFACTS, PATH_METRICS_GRANT_FINETUNE_LARCHER)
)
assert len(df_metrics_grant_finetune_larcher["random_state"].unique()) == S
print(df_metrics_grant_finetune_larcher.shape)
df_metrics_grant_finetune_larcher.head(2)

### Raw dataset

In [ ]:
df_metrics_grant_finetune_raw = pd.read_csv(
    os.path.join(DIR_GRANT_FINETUNE, DIR_ARTIFACTS, PATH_METRICS_GRANT_FINETUNE_RAW)
)
assert len(df_metrics_grant_finetune_raw["random_state"].unique()) == S
print(df_metrics_grant_finetune_raw.shape)
df_metrics_grant_finetune_raw.head(2)

## Feature selection models

### Larcher dataset

In [ ]:
df_metrics_feature_selection_larcher = pd.read_csv(
    os.path.join(
        DIR_FEATURE_SELECTION, DIR_ARTIFACTS, PATH_METRICS_FEATURE_SELECTION_LARCHER
    )
)
assert len(df_metrics_feature_selection_larcher["random_state"].unique()) == S
df_metrics_feature_selection_larcher["model"] = df_metrics_feature_selection_larcher[
    "model"
].replace(
    {"CoxPHSurvivalAnalysis_grant_finetune_T1": "CoxPHSurvivalAnalysis_grant_plus_T1"}
)
print(df_metrics_feature_selection_larcher.shape)
df_metrics_feature_selection_larcher.head(2)

### Raw dataset

In [ ]:
df_metrics_feature_selection_raw = pd.read_csv(
    os.path.join(
        DIR_FEATURE_SELECTION, DIR_ARTIFACTS, PATH_METRICS_FEATURE_SELECTION_RAW
    )
)
assert len(df_metrics_feature_selection_raw["random_state"].unique()) == S
df_metrics_feature_selection_raw["model"] = df_metrics_feature_selection_raw[
    "model"
].replace(
    {"CoxPHSurvivalAnalysis_grant_finetune_T1": "CoxPHSurvivalAnalysis_grant_plus_T1"}
)
print(df_metrics_feature_selection_raw.shape)
df_metrics_feature_selection_raw.head(2)

## Survival models

### Raw dataset

In [ ]:
df_metrics_survival_models = pd.read_csv(
    os.path.join(DIR_SURVIVAL_MODELS, DIR_ARTIFACTS, PATH_METRICS_SURVIVAL_MODELS)
)
assert len(df_metrics_survival_models["random_state"].unique()) == S
print(df_metrics_survival_models.shape)
df_metrics_survival_models.head(2)

# Compare random states

In [ ]:
# Dictionary of DataFrames with their variable names as keys
dataframes = {
    "df_metrics_grant_finetune_raw": df_metrics_grant_finetune_raw,
    "df_metrics_feature_selection_raw": df_metrics_feature_selection_raw,
    "df_metrics_survival_models": df_metrics_survival_models,
    "df_metrics_grant_finetune_larcher": df_metrics_grant_finetune_larcher,
    "df_metrics_feature_selection_larcher": df_metrics_feature_selection_larcher,
}

# Generate all possible combinations of the dataframes
combinations = itertools.combinations(dataframes.items(), 2)

# Compare each combination
for (name1, df1), (name2, df2) in combinations:
    compare_random_states(df1, df2, name1, name2)

# Concat

In [ ]:
time_groups_larcher = {
    "T0 selector": ["RandomSurvivalForest_selector_T0"],
    "T1 benchmark": [
        "CoxPHSurvivalAnalysis_grant_finetune_T1",
        "CoxPHSurvivalAnalysis_grant_plus_T1",
    ],
    "T1 selector": ["RandomSurvivalForest_selector_T1"],
}

color_groups_larcher = {
    "T0 selector": "#ec7d10",
    "T1 benchmark": "#7f7f7f",
    "T1 selector": "#64a6bd",
}


time_groups_raw = {
    "T0 selector": ["RandomSurvivalForest_selector_T0"],
    "T0 models": [
        "SurvivalTree_T0",
        "CoxPHSurvivalAnalysis_T0",
        "CoxnetSurvivalAnalysis_T0",
        "RandomSurvivalForest_T0",
        "ExtraSurvivalTrees_T0",
        "GradientBoostingSurvivalAnalysis_T0",
        "ComponentwiseGradientBoostingSurvivalAnalysis_T0",
    ],
    "T1 benchmark": [
        "CoxPHSurvivalAnalysis_grant_finetune_T1",
        "CoxPHSurvivalAnalysis_grant_plus_T1",
    ],
    "T1 selector": ["RandomSurvivalForest_selector_T1"],
    "T1 models": [
        "SurvivalTree_T1",
        "CoxPHSurvivalAnalysis_T1",
        "CoxnetSurvivalAnalysis_T1",
        "RandomSurvivalForest_T1",
        "ExtraSurvivalTrees_T1",
        "GradientBoostingSurvivalAnalysis_T1",
        "ComponentwiseGradientBoostingSurvivalAnalysis_T1",
    ],
}

color_groups_raw = {
    "T0 selector": "#ec7d10",
    "T0 models": "#fc2f00",
    "T1 benchmark": "#7f7f7f",
    "T1 selector": "#64a6bd",
    "T1 models": "#2b59c3",
}

In [ ]:
# List of T1 models to be removed for T0 analysis
t1_models = (
    time_groups_raw["T1 selector"]
    + time_groups_raw["T1 models"]
    + ["CoxPHSurvivalAnalysis_grant_plus_T1"]
)


# All metrics for larcher dataset
df_metrics_larcher = pd.concat(
    [df_metrics_grant_finetune_larcher, df_metrics_feature_selection_larcher]
)
df_metrics_larcher["dataset"] = "Clinician’s dataset"
# Only T0 DBURI models + T1 GRANT
df_metrics_larcher_t0 = df_metrics_larcher[~df_metrics_larcher["model"].isin(t1_models)]
# All metrics for raw dataset
df_metrics = pd.concat(
    [
        df_metrics_grant_finetune_raw,
        df_metrics_feature_selection_raw,
        df_metrics_survival_models,
    ]
)
df_metrics["dataset"] = "Raw DBURI dataset"
# Only T0 DBURI models + T1 GRANT
df_metrics_t0 = df_metrics[~df_metrics["model"].isin(t1_models)]
# Dataset to compare larcher and raw
df_metrics_vs = pd.concat([df_metrics_larcher, df_metrics])
df_metrics_vs_t0 = pd.concat([df_metrics_larcher_t0, df_metrics_t0])
common_models = (
    df_metrics_vs[df_metrics_vs["dataset"] == "Clinician’s dataset"]["model"]
    .unique()
    .tolist()
)
df_metrics_vs = df_metrics_vs[df_metrics_vs["model"].isin(common_models)]
df_metrics_vs_t0 = df_metrics_vs_t0[df_metrics_vs_t0["model"].isin(common_models)]

# EDA on metrics

(currently only T0 visualizations, to expand to T1 models, remove the suffix _t0 from the variable name

In [ ]:
dict_fig = dict()

## Larcher vs raw dataset at T1 + T0

In [ ]:
ax = plot_metric_boxplot(
    df_metrics_vs,
    "concordance_index_censored",
    time_groups_larcher,
    color_groups_larcher,
    hue_column="dataset",
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_vs_larcher_t1_t0_concordance_index_censored.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


In [ ]:
ax = plot_metric_boxplot(
    df_metrics_vs,
    "concordance_index_ipcw",
    time_groups_larcher,
    color_groups_larcher,
    hue_column="dataset",
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_vs_larcher_t1_t0_concordance_index_ipcw.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


In [ ]:
ax = plot_metric_boxplot(
    df_metrics_vs,
    "integrated_brier_score",
    time_groups_larcher,
    color_groups_larcher,
    hue_column="dataset",
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_vs_larcher_t1_t0_integrated_brier_score.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


In [ ]:
ax = plot_metric_boxplot(
    df_metrics_vs,
    "mean_cumulative_dynamic_auc",
    time_groups_larcher,
    color_groups_larcher,
    hue_column="dataset",
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_vs_larcher_t1_t0_mean_cumulative_dynamic_auc.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


## Larcher vs raw dataset at T0

In [ ]:
ax = plot_metric_boxplot(
    df_metrics_vs_t0,
    "concordance_index_censored",
    time_groups_larcher,
    color_groups_larcher,
    hue_column="dataset",
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_vs_larcher_t0_concordance_index_censored.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


In [ ]:
ax = plot_metric_boxplot(
    df_metrics_vs_t0,
    "concordance_index_ipcw",
    time_groups_larcher,
    color_groups_larcher,
    hue_column="dataset",
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_vs_larcher_t0_concordance_index_ipcw.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


In [ ]:
ax = plot_metric_boxplot(
    df_metrics_vs_t0,
    "integrated_brier_score",
    time_groups_larcher,
    color_groups_larcher,
    hue_column="dataset",
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_vs_larcher_t0_integrated_brier_score.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


In [ ]:
ax = plot_metric_boxplot(
    df_metrics_vs_t0,
    "mean_cumulative_dynamic_auc",
    time_groups_larcher,
    color_groups_larcher,
    hue_column="dataset",
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_vs_larcher_t0_mean_cumulative_dynamic_auc.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


### Multiple plots in 1 figure

In [ ]:
metrics = [
    "concordance_index_censored",
    "concordance_index_ipcw",
    "integrated_brier_score",
]
hue_column = "dataset"


# Determine the number of metrics to set up subplots
num_metrics = len(metrics)
cols = 3  # Number of columns in the subplot grid
rows = (num_metrics + cols - 1) // cols  # Calculate the number of rows needed
# Create the composite figure with subplots
fig, axes = plt.subplots(rows, cols, figsize=(cols * 8, rows * 9), squeeze=False)
axes_flat = axes.flatten()

dict_fig = {}
for idx, metric in tqdm(enumerate(metrics)):
    ax = axes_flat[idx]
    plot_metric_boxplot(
        df_metrics=df_metrics_vs_t0,
        metric=metric,
        time_groups=time_groups_larcher,
        color_groups=color_groups_larcher,
        hue_column=hue_column,
        ax=ax,
    )
    if idx == len(metrics) - 1:
        # Add legend to the last ax
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles,
            labels,
            title=hue_column,
            loc="upper left",
            bbox_to_anchor=(1.05, 1),
            ncol=1,
        )
    else:
        # Remove legend
        ax.get_legend().remove()
    dict_fig[metric] = ax

# Hide any unused subplots
for idx in range(len(metrics), len(axes_flat)):
    fig.delaxes(axes_flat[idx])
# Adjust layout to accommodate the suptitle and the single legend
plt.tight_layout(rect=[0, 0, 0.85, 0.95])

# Save as vector PDF
os.makedirs(DIR_FIGURES, exist_ok=True)
fig.savefig(
    os.path.join(DIR_FIGURES, "raw_vs_larcher_mccv_internal_val.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


## All models on raw dataset at T1 + T0

In [ ]:
ax = plot_metric_boxplot(
    df_metrics, "concordance_index_censored", time_groups_raw, color_groups_raw
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_t1_t0_concordance_index_censored.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


In [ ]:
ax = plot_metric_boxplot(
    df_metrics, "concordance_index_ipcw", time_groups_raw, color_groups_raw
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_t1_t0_concordance_index_ipcw.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


In [ ]:
ax = plot_metric_boxplot(
    df_metrics, "integrated_brier_score", time_groups_raw, color_groups_raw
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_t1_t0_integrated_brier_score.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


In [ ]:
ax = plot_metric_boxplot(
    df_metrics, "mean_cumulative_dynamic_auc", time_groups_raw, color_groups_raw
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_t1_t0_mean_cumulative_dynamic_auc.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


## All models on raw dataset at T0

In [ ]:
ax = plot_metric_boxplot(
    df_metrics_t0, "concordance_index_censored", time_groups_raw, color_groups_raw
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_t0_concordance_index_censored.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


In [ ]:
ax = plot_metric_boxplot(
    df_metrics_t0, "concordance_index_ipcw", time_groups_raw, color_groups_raw
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_t0_concordance_index_ipcw.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


In [ ]:
ax = plot_metric_boxplot(
    df_metrics_t0, "integrated_brier_score", time_groups_raw, color_groups_raw
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_t0_integrated_brier_score.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


In [ ]:
ax = plot_metric_boxplot(
    df_metrics_t0, "mean_cumulative_dynamic_auc", time_groups_raw, color_groups_raw
)
ax.get_figure().savefig(
    os.path.join(DIR_FIGURES, "raw_t0_mean_cumulative_dynamic_auc.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


### Multiple plots in 1 figure

In [ ]:
metrics = [
    "concordance_index_censored",
    "concordance_index_ipcw",
    "integrated_brier_score",
]
hue_column = "dataset"


# Determine the number of metrics to set up subplots
num_metrics = len(metrics)
cols = 3  # Number of columns in the subplot grid
rows = (num_metrics + cols - 1) // cols  # Calculate the number of rows needed
# Create the composite figure with subplots
fig, axes = plt.subplots(rows, cols, figsize=(cols * 8, rows * 9), squeeze=False)
axes_flat = axes.flatten()

dict_fig = {}
for idx, metric in tqdm(enumerate(metrics)):
    ax = axes_flat[idx]
    plot_metric_boxplot(
        df_metrics=df_metrics_t0,
        metric=metric,
        time_groups=time_groups_raw,
        color_groups=color_groups_raw,
        hue_column=None,
        ax=ax,
    )
    if idx == len(metrics) - 1:
        # Add legend to the last ax
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles,
            labels,
            title=hue_column,
            loc="upper left",
            bbox_to_anchor=(1.05, 1),
            ncol=1,
        )
    else:
        # Remove legend
        ax.get_legend().remove()
    dict_fig[metric] = ax

# Hide any unused subplots
for idx in range(len(metrics), len(axes_flat)):
    fig.delaxes(axes_flat[idx])
# Adjust layout to accommodate the suptitle and the single legend
plt.tight_layout(rect=[0, 0, 0.85, 0.95])

# Save as vector PDF
os.makedirs(DIR_FIGURES, exist_ok=True)
fig.savefig(
    os.path.join(DIR_FIGURES, "raw_t0_mccv_internal_val.pdf"),
    format="pdf",
    bbox_inches="tight",
)
plt.show()


# Number of features

## T1 + T0

In [ ]:
to_drop = time_groups_raw["T0 selector"] + time_groups_raw["T1 selector"]
df_plot = df_metrics_t0[~df_metrics_t0["model"].isin(to_drop)]
plot_features_violin(
    df_plot,
    time_groups_raw,
    color_groups_raw,
    figsize=(10, 8),
    save_path=os.path.join(DIR_FIGURES, "raw_t0_features_violin.pdf"),
)


## T1

In [ ]:
to_drop = time_groups_raw["T0 selector"] + time_groups_raw["T1 selector"]
df_plot = df_metrics[~df_metrics["model"].isin(to_drop)]
plot_features_violin(
    df_plot,
    time_groups_raw,
    color_groups_raw,
    figsize=(10, 8),
    save_path=os.path.join(DIR_FIGURES, "raw_t1_t0_features_violin.pdf"),
)


# Aggregate by model

## Larcher vs raw dataset

In [ ]:
assert len(df_metrics_vs["random_state"].unique()) == S
# Aggregate the metrics to get both mean and standard deviation
df_metrics_agg = (
    df_metrics_vs.groupby(["model", "dataset"])[
        [
            "best_performance_tuning",
            "concordance_index_censored",
            "concordance_index_ipcw",
            "mean_cumulative_dynamic_auc",
            "integrated_brier_score",
            "n_features_in",
        ]
    ].agg(["mean"])
).droplevel(1, axis=1)
# Calculate the delta tuning validation using the mean values
df_metrics_agg[("delta_tuning_validation")] = (
    -df_metrics_agg[("best_performance_tuning")]
    - df_metrics_agg[("concordance_index_censored")]
)
# Drop the best_performance_tuning columns (both mean and std)
df_metrics_agg.drop(("best_performance_tuning"), axis=1, inplace=True)
# Round the values to 5 decimal places and sort by concordance_index_ipcw_mean
df_metrics_agg_rounded = df_metrics_agg.round(3).sort_values(
    ("concordance_index_ipcw"), ascending=False
)
df_metrics_agg_rounded

## All models on raw dataset

In [ ]:
assert len(df_metrics["random_state"].unique()) == S
# Aggregate the metrics to get both mean and standard deviation
df_metrics_agg = (
    df_metrics.groupby(["model", "dataset"])[
        [
            "best_performance_tuning",
            "concordance_index_censored",
            "concordance_index_ipcw",
            "mean_cumulative_dynamic_auc",
            "integrated_brier_score",
            "n_features_in",
        ]
    ].agg(["mean"])
).droplevel(1, axis=1)
# Calculate the delta tuning validation using the mean values
df_metrics_agg[("delta_tuning_validation")] = (
    -df_metrics_agg[("best_performance_tuning")]
    - df_metrics_agg[("concordance_index_censored")]
)
# Drop the best_performance_tuning columns (both mean and std)
df_metrics_agg.drop(("best_performance_tuning"), axis=1, inplace=True)
# Round the values to 5 decimal places and sort by concordance_index_ipcw_mean
df_metrics_agg_rounded = df_metrics_agg.round(5).sort_values(
    ("concordance_index_ipcw"), ascending=False
)
df_metrics_agg_rounded

`delta_tuning_validation` is an indicator for overfitting.

# Model of choice

In order to spot the best model, we apply the following :
- Remove the models with - on average over the simulations - more than 13 features as obtimal (hard to apply in a clinical environment)
- Normalization: Normalize each metric so that they are on the same scale, between 0 and 1. We'll use Min-Max normalization. This allows us to combine them meaningfully.
- Weight Assignment: Assign weights to each normalized metric based on their importance (our KPI is c-index). The weights are:
    - Concordance Index (IPCW) Mean: 0.5
    - Integrated Brier Score Mean: 0.25
    - Number of Features (n_features_in) Mean: 0.25

Note: For the metrics where "lower is better" (such as Integrated Brier Score Mean, Number of Features, and Delta Tuning Validation Mean), we will invert the normalized score by subtracting it from 1.
- Weighted Average Calculation: Compute the weighted average of these normalized metrics for each model.
- Ranking Models: Rank the models based on the calculated weighted average to identify the best model.

## T1

In [ ]:
timepoint = "T1"
n_max = 13

# Filter feature selectors and GRANT models
df_metrics_best = df_metrics_agg_rounded.reset_index()
df_metrics_best = df_metrics_best[
    (df_metrics_best["model"].str.contains(timepoint))
    & (~df_metrics_best["model"].str.contains("_selector"))
    & (~df_metrics_best["model"].str.contains("_grant"))
]

# Exclude models with too many features
df_metrics_best = df_metrics_best[df_metrics_best["n_features_in"] < n_max]


# Normalization
def min_max_normalize(series):
    return (series - series.min()) / (series.max() - series.min())


# Normalize each metric
df_metrics_best["concordance_index_ipcw_norm"] = min_max_normalize(
    df_metrics_best["concordance_index_ipcw"]
)
df_metrics_best["integrated_brier_score_norm"] = 1 - min_max_normalize(
    df_metrics_best["integrated_brier_score"]
)  # Lower is better
df_metrics_best["n_features_in_norm"] = 1 - min_max_normalize(
    df_metrics_best["n_features_in"]
)  # Lower is better

# Weights for each metric
weights = {
    "concordance_index_ipcw_norm": 0.5,
    "integrated_brier_score_norm": 0.2,
    "n_features_in_norm": 0.2,
}

# Calculate weighted average score
df_metrics_best["weighted_average_score"] = (
    df_metrics_best["concordance_index_ipcw_norm"]
    * weights["concordance_index_ipcw_norm"]
    + df_metrics_best["integrated_brier_score_norm"]
    * weights["integrated_brier_score_norm"]
    + df_metrics_best["n_features_in_norm"] * weights["n_features_in_norm"]
)

# Drop intermediate columns
df_metrics_best = df_metrics_best.drop(
    [
        "concordance_index_ipcw_norm",
        "integrated_brier_score_norm",
        "n_features_in_norm",
    ],
    axis=1,
)

# Sort by weighted average score
model_t1 = (
    df_metrics_best.sort_values(by="weighted_average_score", ascending=False)
    .head(1)["model"]
    .values[0]
)
print("Best model according to weighted average", model_t1)
df_metrics_best.sort_values(by="weighted_average_score", ascending=False).head(3)

In [ ]:
df_metrics[(df_metrics["model"] == model_t1)][
    "n_features_in"
].value_counts().sort_index()

In [ ]:
df_metrics[df_metrics["model"] == model_t1].groupby("n_features_in")[
    "concordance_index_ipcw"
].mean()

## T0

In [ ]:
timepoint = "T0"
n_max = 13

# Filter feature selectors and GRANT models
df_metrics_best = df_metrics_agg_rounded.reset_index()
df_metrics_best = df_metrics_best[
    (df_metrics_best["model"].str.contains(timepoint))
    & (~df_metrics_best["model"].str.contains("_selector"))
    & (~df_metrics_best["model"].str.contains("_grant"))
]

# Exclude models with too many features
df_metrics_best = df_metrics_best[df_metrics_best["n_features_in"] < n_max]


# Normalization
def min_max_normalize(series):
    return (series - series.min()) / (series.max() - series.min())


# Normalize each metric
df_metrics_best["concordance_index_ipcw_norm"] = min_max_normalize(
    df_metrics_best["concordance_index_ipcw"]
)
df_metrics_best["integrated_brier_score_norm"] = 1 - min_max_normalize(
    df_metrics_best["integrated_brier_score"]
)  # Lower is better
df_metrics_best["n_features_in_norm"] = 1 - min_max_normalize(
    df_metrics_best["n_features_in"]
)  # Lower is better
df_metrics_best["mean_cumulative_dynamic_auc_norm"] = min_max_normalize(
    df_metrics_best["mean_cumulative_dynamic_auc"]
)
df_metrics_best["delta_tuning_validation_norm"] = 1 - min_max_normalize(
    df_metrics_best["delta_tuning_validation"].fillna(
        df_metrics_best["delta_tuning_validation"].max()
    )
)  # Lower is better

# Weights for each metric
weights = {
    "concordance_index_ipcw_norm": 0.5,
    "integrated_brier_score_norm": 0.25,
    "n_features_in_norm": 0.25,
}

# Calculate weighted average score
df_metrics_best["weighted_average_score"] = (
    df_metrics_best["concordance_index_ipcw_norm"]
    * weights["concordance_index_ipcw_norm"]
    + df_metrics_best["integrated_brier_score_norm"]
    * weights["integrated_brier_score_norm"]
    + df_metrics_best["n_features_in_norm"] * weights["n_features_in_norm"]
)

# Drop intermediate columns
df_metrics_best = df_metrics_best.drop(
    [
        "concordance_index_ipcw_norm",
        "integrated_brier_score_norm",
        "n_features_in_norm",
        "mean_cumulative_dynamic_auc_norm",
        "delta_tuning_validation_norm",
    ],
    axis=1,
)

# Sort by weighted average score
model_t0 = (
    df_metrics_best.sort_values(by="weighted_average_score", ascending=False)
    .head(1)["model"]
    .values[0]
)
print("Best model according to weighted average", model_t0)
df_metrics_best.sort_values(by="weighted_average_score", ascending=False).head(3)

In [ ]:
df_metrics[(df_metrics["model"] == model_t0)][
    "n_features_in"
].value_counts().sort_index()

In [ ]:
df_metrics[df_metrics["model"] == model_t0].groupby("n_features_in")[
    "concordance_index_ipcw"
].mean()